# Titanic EDA

Goal: explore the Titanic dataset and understand which features may help predict passenger survival.

Dataset: Kaggle Titanic - Machine Learning from Disaster.

In [63]:
import pandas as pd
from IPython.display import display

train = pd.read_csv("../../../data/titanic/train.csv")
test = pd.read_csv("../../../data/titanic/test.csv")



## 1. Basic dataset structure

In this section, I check the columns, table sizes, data types, and missing values.

In [64]:
display(train.columns)
display(test.columns)
display((train.shape, test.shape))


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

((891, 12), (418, 11))

In [65]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [66]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


In [67]:
missing_values = pd.DataFrame({
    "train_missing": train.isna().sum(),
    "test_missing": test.isna().sum()
})

display(missing_values)


,train_missing,test_missing
Age,177,86.0
Cabin,687,327.0
Embarked,2,0.0
Fare,0,1.0
Name,0,0.0
Parch,0,0.0
PassengerId,0,0.0
Pclass,0,0.0
Sex,0,0.0
SibSp,0,0.0


In [68]:
survival_stats = pd.DataFrame({
    "count": train["Survived"].value_counts(),
    "share": train["Survived"].value_counts(normalize=True)
})
display(survival_stats)

survival_by_sex = train.groupby("Sex")["Survived"].agg(["count", "mean"])
display(survival_by_sex)

survival_by_class = train.groupby("Pclass")["Survived"].agg(["count", "mean"])
display(survival_by_class)

survival_by_sex_and_class = train.groupby(["Pclass","Sex"])["Survived"].agg(["count", "mean"])
display(survival_by_sex_and_class)

,count,share
Survived,,
0,549,0.616162
1,342,0.383838


,count,mean
Sex,,
female,314,0.742038
male,577,0.188908


,count,mean
Pclass,,
1,216,0.629630
2,184,0.472826
3,491,0.242363


count      mean
Pclass Sex                    
1      female     94  0.968085
       male      122  0.368852
2      female     76  0.921053
       male      108  0.157407
3      female    144  0.500000
       male      347  0.135447

## 2. Numeric features

I will inspect numeric features

In [69]:
display(train[["Age", "Fare", "SibSp", "Parch"]].describe())


train["AgeGroup"] = pd.cut(
    train["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

survival_by_age_group = train.groupby("AgeGroup", observed=True)["Survived"].agg(["count", "mean"])
display(survival_by_age_group)

train["FareGroup"] = pd.qcut(
    train["Fare"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

survival_by_fare_group = train.groupby("FareGroup", observed=True)["Survived"].agg(["count", "mean"])
display(survival_by_fare_group)

survival_by_embarked = train.groupby("Embarked")["Survived"].agg(["count", "mean"])
display(survival_by_embarked)

train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
survival_by_family_size = train.groupby("FamilySize")["Survived"].agg(["count", "mean"])
display(survival_by_family_size)

train["IsAlone"] = (train["FamilySize"] == 1)
survival_by_is_alone = train.groupby("IsAlone")["Survived"].agg(["count", "mean"])
display(survival_by_is_alone)

train["HasCabin"] = train["Cabin"].notna()
survival_by_has_cabin = train.groupby("HasCabin")["Survived"].agg(["count", "mean"])
display(survival_by_has_cabin)



,Age,Fare,SibSp,Parch
count,714.000000,891.000000,891.000000,891.000000
mean,29.699118,32.204208,0.523008,0.381594
std,14.526497,49.693429,1.102743,0.806057
min,0.420000,0.000000,0.000000,0.000000
25%,20.125000,7.910400,0.000000,0.000000
50%,28.000000,14.454200,0.000000,0.000000
75%,38.000000,31.000000,1.000000,0.000000
max,80.000000,512.329200,8.000000,6.000000


,count,mean
AgeGroup,,
Child,69,0.579710
Teen,70,0.428571
Young Adult,358,0.382682
Adult,195,0.400000
Senior,22,0.227273


,count,mean
FareGroup,,
Low,223,0.197309
Medium,224,0.303571
High,222,0.454955
Very High,222,0.581081


,count,mean
Embarked,,
C,168,0.553571
Q,77,0.389610
S,644,0.336957


,count,mean
FamilySize,,
1,537,0.303538
2,161,0.552795
3,102,0.578431
4,29,0.724138
5,15,0.200000
6,22,0.136364
7,12,0.333333
8,6,0.000000
11,7,0.000000


,count,mean
IsAlone,,
False,354,0.505650
True,537,0.303538


,count,mean
HasCabin,,
False,687,0.299854
True,204,0.666667


,Name,SibSp,Parch,Ticket,Survived
159,"Sage, Master. Thomas Henry",8,2,CA. 2343,0
180,"Sage, Miss. Constance Gladys",8,2,CA. 2343,0
201,"Sage, Mr. Frederick",8,2,CA. 2343,0
324,"Sage, Mr. George John Jr",8,2,CA. 2343,0
792,"Sage, Miss. Stella Anna",8,2,CA. 2343,0
846,"Sage, Mr. Douglas Bullen",8,2,CA. 2343,0
863,"Sage, Miss. Dorothy Edith ""Dolly""",8,2,CA. 2343,0
